# Phase 6: 構造化RAG 完全評価 (v6.2)

**Phase 6.2 改善対象**:
- spatial_proximity: -6.9pt → +10pt以上（最近傍検索）
- advanced_sensitivity: -11.3pt → +10pt以上（感度分析）

**作成日**: 2026-01-23

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers tqdm pandas
print("パッケージインストール完了")

In [ ]:
# 1.2 GPU確認・メモリ管理
import torch, gc

def print_memory():
    if torch.cuda.is_available(): print(f"GPU VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")
    import psutil; print(f"RAM: {psutil.virtual_memory().percent}%")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")
print_memory()

In [ ]:
# 1.3 Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR, RESULTS_DIR = f"{BASE_DIR}/data", f"{BASE_DIR}/results"
for d in [DATA_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)
sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.4 モデル設定
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
print(f"LLM: {LLM_MODEL}\nEmbedding: {EMBEDDING_MODEL}")

## Section 2: データ読み込み

In [ ]:
# 2.1 POIデータ読み込み
import json
from collections import Counter

with open(f"{DATA_DIR}/poi_documents.json", "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

all_pois = []
for doc in poi_documents:
    poi = doc["metadata"].copy() if "metadata" in doc else doc.copy()
    poi["content"] = doc.get("content", "")
    all_pois.append(poi)
print(f"POIデータ: {len(all_pois)}件")

In [ ]:
# 2.2 Phase 6モジュール読み込み
from src.geo_utils import (enrich_all_pois, get_nearest_pois, filter_by_radius,
    compare_by_radius, generate_proximity_context, generate_sensitivity_context)
from src.aggregator import (compare_east_west, get_top_categories,
    analyze_category_by_direction, filter_by_category)
from src.structured_rag_system import analyze_question

enriched_pois = enrich_all_pois(all_pois)
print(f"空間情報追加完了: {len(enriched_pois)}件")

# Phase 6.2機能テスト
nearest = get_nearest_pois(enriched_pois, category="カフェ", top_n=3)
print(f"\n最寄りカフェTOP3:")
for i, c in enumerate(nearest, 1): print(f"  {i}. {c['name']} - {c['distance_from_station']:.0f}m")

comp = compare_by_radius(enriched_pois, 300, 500, "カフェ")
print(f"\n半径比較: {comp.to_japanese()}")

In [ ]:
# 2.3 テストケース読み込み
try:
    from src.test_cases_v2 import TEST_CASES_V2, get_test_cases_by_level, get_test_cases_by_subcategory
    print(f"テストケース: {len(TEST_CASES_V2)}件")
    for level in ["L1","L2","L3","L4","L5"]:
        print(f"  {level}: {len(get_test_cases_by_level(level))}件")
except ImportError as e:
    print(f"エラー: {e}")
    TEST_CASES_V2 = None

## Section 3: モデルセットアップ

In [ ]:
# 3.1 Embeddingモデル
from langchain_huggingface import HuggingFaceEmbeddings
print("Embeddingモデルロード中...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL,
    model_kwargs={'device': DEVICE}, encode_kwargs={'normalize_embeddings': True})
print("完了"); print_memory()

In [ ]:
# 3.2 ベクトルストア構築（ローカル）
from langchain_chroma import Chroma
from langchain_core.documents import Document
print("ベクトルストア構築中...")
documents = [Document(page_content=p.get("content",f"{p.get('name','')}"), metadata=p) for p in poi_documents]
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name="poi_phase6")
print(f"完了: {len(documents)}件"); print_memory()

In [ ]:
# 3.3 Embedding解放
del embeddings; clear_memory(); print_memory()

In [ ]:
# 3.4 LLMモデルロード
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
print(f"LLMロード中: {LLM_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, quantization_config=quantization_config,
    device_map="auto", trust_remote_code=True, low_cpu_mem_usage=True)
print("完了"); print_memory()

## Section 4: RAGシステム (Phase 6.2)

In [ ]:
# 4.1 構造化RAGシステム（Phase 6.2対応）
import time

class StructuredRAGEvaluator:
    def __init__(self, model, tokenizer, vectorstore, all_pois):
        self.model, self.tokenizer, self.vectorstore, self.all_pois = model, tokenizer, vectorstore, all_pois
        self.system_prompt = """あなたは渋谷エリアの地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
数値データがある場合は具体的な数字を使って回答してください。"""
    
    def _generate(self, prompt, max_tokens=512):
        messages = [{"role":"system","content":self.system_prompt},{"role":"user","content":prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.1,
                do_sample=True, pad_token_id=self.tokenizer.eos_token_id)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response.split("assistant")[-1].strip() if "assistant" in response.lower() else response
    
    def _build_context(self, question, analysis):
        parts = []
        cat = analysis.subcategories[0] if analysis.subcategories else None
        
        if analysis.requires_proximity and cat:
            parts.append(generate_proximity_context(self.all_pois, cat, top_n=5))
        elif analysis.requires_sensitivity and cat:
            r1, r2 = analysis.sensitivity_radii if analysis.sensitivity_radii else (300, 500)
            parts.append(generate_sensitivity_context(self.all_pois, cat, r1, r2))
            comp = compare_by_radius(self.all_pois, r1, r2, cat)
            if comp.count1 > 0:
                conclusion = "半径を変えると件数が大きく変化します。" if comp.ratio >= 1.5 else "半径を変えても結論は大きく変わりません。"
                parts.append(f"\n【結論】\n{conclusion}")
        elif analysis.requires_comparison and "東" in question and "西" in question:
            result = compare_east_west(self.all_pois, cat)
            parts.append(f"【東西比較】\n{result.to_japanese()}")
            if cat:
                detail = analyze_category_by_direction(self.all_pois, cat)
                parts.append("\n方向別:" + "".join([f"\n  {d}: {c}件" for d,c in detail['by_direction'].items()]))
        elif analysis.requires_aggregation:
            if cat:
                filtered = filter_by_category(self.all_pois, cat)
                parts.append(f"【{cat}の集計】\n総数: {len(filtered)}件")
            else:
                top = get_top_categories(self.all_pois, 5)
                parts.append("【カテゴリランキング】" + "".join([f"\n  {i}. {c.category}: {c.count}件" for i,c in enumerate(top,1)]))
        
        if not parts:
            try:
                results = self.vectorstore.similarity_search(question, k=5)
                if results:
                    parts.append("【関連POI】" + "".join([f"\n- {r.metadata.get('name','?')} ({r.metadata.get('category','')})" for r in results]))
            except: pass
        return "\n".join(parts)
    
    def query(self, question):
        start = time.time()
        analysis = analyze_question(question)
        context = self._build_context(question, analysis)
        prompt = f"以下の情報を参考に回答してください。\n\n{context}\n\n【質問】\n{question}\n\n【回答】"
        answer = self._generate(prompt)
        return {"answer": answer, "analysis": analysis.to_dict(), "context": context, "time_sec": round(time.time()-start, 2)}

rag_system = StructuredRAGEvaluator(model, tokenizer, vectorstore, enriched_pois)
print("RAGシステム初期化完了（Phase 6.2）")

In [ ]:
# 4.2 動作確認
print("=== Phase 6.2 動作確認 ===")
q1 = "渋谷駅に最も近いコンビニはどこですか？"
r1 = rag_system.query(q1)
print(f"\n[近接性] Q: {q1}\nType: {r1['analysis']['question_type']}\nA: {r1['answer'][:150]}...")

q2 = "渋谷駅周辺はカフェが多いという結論は、半径を500mから300mに変えても成立しますか？"
r2 = rag_system.query(q2)
print(f"\n[感度分析] Q: {q2[:50]}...\nType: {r2['analysis']['question_type']}\nA: {r2['answer'][:150]}...")

## Section 5: 評価実行

In [ ]:
# 5.1 評価関数
import re
from tqdm import tqdm

def evaluate_response(response, tc):
    answer = response.get("answer", "")
    kws = tc.expected_keywords if hasattr(tc, 'expected_keywords') else []
    kw_score = sum(1 for kw in kws if kw in answer) / len(kws) * 100 if kws else 50
    coord_score = 100 if "35." in answer and "139." in answer else 0
    num_score = 100 if re.findall(r'\d+', answer) else 0
    total = kw_score * 0.4 + coord_score * 0.3 + num_score * 0.3
    return {"keyword_score": kw_score, "coord_score": coord_score, "number_score": num_score, "total_score": round(total, 1)}

def run_evaluation(rag, cases, max_cases=None):
    results = []
    for tc in tqdm(cases[:max_cases] if max_cases else cases, desc="評価中"):
        try:
            resp = rag.query(tc.prompt)
            ev = evaluate_response(resp, tc)
            results.append({"id": tc.id, "level": tc.level, "category": tc.category, "subcategory": tc.subcategory,
                "prompt": tc.prompt, "answer": resp["answer"][:500], "time_sec": resp.get("time_sec",0),
                "analysis": resp.get("analysis",{}), "scores": ev})
            clear_memory()
        except Exception as e:
            print(f"Error ({tc.id}): {e}")
            results.append({"id": tc.id, "level": tc.level, "error": str(e)})
    return results

In [ ]:
# 5.2 重点テスト
if TEST_CASES_V2:
    print("=== Phase 6.2 重点テスト ===")
    for subcat in ["spatial_proximity", "advanced_sensitivity"]:
        cases = get_test_cases_by_subcategory(subcat)
        print(f"\n{subcat}: {len(cases)}件")
        results = run_evaluation(rag_system, cases)
        for r in results:
            if "error" not in r:
                print(f"  {r['id']}: {r['scores']['total_score']:.1f}pt | {r['analysis'].get('question_type','?')}")

In [ ]:
# 5.3 全テスト実行
if TEST_CASES_V2:
    print(f"全テスト実行: {len(TEST_CASES_V2)}件（約30-40分）")
    all_results_rag = run_evaluation(rag_system, TEST_CASES_V2)
    print(f"完了: {len(all_results_rag)}件")

## Section 6: 結果分析

In [ ]:
# 6.1 分析
def analyze_results(results):
    valid = [r for r in results if "error" not in r]
    level_scores, subcat_scores = {}, {}
    for level in ["L1","L2","L3","L4","L5"]:
        lr = [r for r in valid if r["level"]==level]
        if lr: level_scores[level] = {"count":len(lr), "avg":round(sum(r["scores"]["total_score"] for r in lr)/len(lr),1)}
    for r in valid:
        sc = r["subcategory"]
        subcat_scores.setdefault(sc, []).append(r["scores"]["total_score"])
    subcat_avg = {sc: round(sum(s)/len(s),1) for sc, s in subcat_scores.items()}
    all_scores = [r["scores"]["total_score"] for r in valid]
    times = [r["time_sec"] for r in valid if "time_sec" in r]
    return {"overall": {"count":len(all_scores), "avg":round(sum(all_scores)/len(all_scores),1) if all_scores else 0},
            "by_level": level_scores, "by_subcategory": subcat_avg, "avg_time_sec": round(sum(times)/len(times),1) if times else 0}

if 'all_results_rag' in dir():
    analysis = analyze_results(all_results_rag)
    print(f"=== Phase 6.2 結果 ===")
    print(f"全体: {analysis['overall']['avg']}pt / {analysis['avg_time_sec']}秒")
    print("\nレベル別:"); [print(f"  {l}: {s['avg']}pt") for l,s in analysis['by_level'].items()]
    print("\nサブカテゴリ別:"); [print(f"  {sc}: {avg}pt") for sc,avg in sorted(analysis['by_subcategory'].items(), key=lambda x:x[1], reverse=True)]

In [ ]:
# 6.2 Phase比較
phase5 = {"basic_location":71.7,"basic_category":58.3,"spatial_proximity":61.7,"spatial_density":65.0,
    "spatial_comparison":51.4,"constraint_single":53.3,"constraint_multi":53.3,"decision_location":63.3,
    "decision_business":63.3,"advanced_sensitivity":60.0,"advanced_comparison":59.5,"advanced_uncertainty":66.7}
phase61 = {"basic_location":96.8,"basic_category":81.3,"spatial_proximity":54.8,"spatial_density":62.8,
    "spatial_comparison":76.0,"constraint_single":51.3,"constraint_multi":80.0,"decision_location":57.6,
    "decision_business":83.5,"advanced_sensitivity":48.7,"advanced_comparison":64.4,"advanced_uncertainty":67.3}

if 'analysis' in dir():
    print(f"\n{'サブカテゴリ':<25} {'P5':>6} {'P6.1':>6} {'P6.2':>6} {'Δ6.2':>6}")
    print("-"*55)
    for sc in phase5:
        p5, p61, p62 = phase5[sc], phase61.get(sc,0), analysis['by_subcategory'].get(sc,0)
        m = "✅" if p62>p61 else "❌" if p62<p61 else "-"
        print(f"{sc:<25} {p5:>6.1f} {p61:>6.1f} {p62:>6.1f} {p62-p61:>+6.1f} {m}")

## Section 7: 結果保存

In [ ]:
# 7.1 JSON保存
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if 'all_results_rag' in dir() and 'analysis' in dir():
    data = {"timestamp":timestamp, "phase":"6.2", "model":LLM_MODEL, "poi_count":len(enriched_pois),
            "analysis":analysis, "phase5":phase5, "phase61":phase61, "results":all_results_rag}
    path = f"{RESULTS_DIR}/phase62_eval_{timestamp}.json"
    with open(path, "w", encoding="utf-8") as f: json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"保存: {path}")

In [ ]:
# 7.2 レポート生成
if 'analysis' in dir():
    prox = analysis['by_subcategory'].get('spatial_proximity',0)
    sens = analysis['by_subcategory'].get('advanced_sensitivity',0)
    report = f"""# Phase 6.2 評価レポート
**日時**: {timestamp} | **モデル**: {LLM_MODEL} | **POI**: {len(enriched_pois)}件

## 全体: {analysis['overall']['avg']}pt / {analysis['avg_time_sec']}秒

## Phase 6.2 重点改善
| 項目 | P6.1 | P6.2 | 改善 |
|------|------|------|------|
| spatial_proximity | 54.8pt | {prox}pt | {prox-54.8:+.1f}pt |
| advanced_sensitivity | 48.7pt | {sens}pt | {sens-48.7:+.1f}pt |
"""
    rpath = f"{RESULTS_DIR}/phase62_report_{timestamp}.md"
    with open(rpath, "w", encoding="utf-8") as f: f.write(report)
    print(f"レポート: {rpath}\n\n{report}")